# Text to speech (TTS)
- TTS is one of the coolest subtasks in the machine learning and AI space, and one of the fastest growing fields (audio dev) in general today! In my experience, aside from LLM models, I see many TTS (and other audio-task related models) going obsolete and replacing each other by iterating into better models, at an extremely fast pace! It's truly extremely exciting.

## How does the TTS architecture work?
Here is a very high-level outline of how TTS works:

### The "frontend"
  - Text-processing 
    - Normalization (expanding abbreviation, handling special characters, converting numbers): to clean the text
  - Tokenization/phonemization: Processed text needs to get converted into phonemes, which are the distinct   sounds that make up words. Very important step when it comes to accurately capturing how the text should sound when spoken out loud.
    - Example: The word "cat" is represented by the phonemes /k/, /æ/, /t/.
  - Prosody Modeling: Helps make the speech sound more normal and less robotic. This step essentially takes away the feeling that the audio was created by a TTS service (for example, using the speaker option in Google Translate always creates audio that very obviously sounds like it was made using a TTS service). Adds rhythm, stress, intonation, to generated audio to make it sound close to human speech.
### The "backend"
- Speech Synthesis: The system generates the audio waveforms from the processed text, creating the spoken voice that users hear. This is the final output.

## Delving deep
- Now we're going to go through each step of the text-to-speech process in great depth.

### Text preprocessing
- This is a relatively simple step. You essentially process the text into a version that makes transforms it from original raw input text into a format that is suitable for speech suitable.
- Below is an example of how to normalize text, including some of the most common techniques:
    - Abbreviation expansion (e.g., “Dr.” → “doctor”)
    - Number conversion (e.g., “50.00” → “fifty”)
    - Special characters handling (e.g., “$50.00” → “fifty dollars”)

- Here is an example of a text-preprocessing function:
```python
def preprocess_for_tts(text):
    # Basic abbreviations dictionary
    abbreviations = {
        'dr.': 'doctor',
    }
   
    # Convert text to lowercase for easier matching
    words = text.lower().split()
    processed_words = []
   
    for word in words:
        # Handle dollar amounts
        if word.startswith('$'):
            number = float(word.replace('$', ''))
            processed_words.append(num2words(number) + " dollars")
        # Expand abbreviations
        elif word in abbreviations:
            processed_words.append(abbreviations[word])
        # Convert plain numbers
        elif word.replace('.', '').isdigit():
            processed_words.append(num2words(float(word)))
        else:
            processed_words.append(word)
   
    return ' '.join(processed_words)

# Example
text = "Dr. Smith paid $50.00"
processed_text = preprocess_for_tts(text)

print(f"Processed: {processed_text}")
```

### Phonetic Conversion
- Phonemes are the smallest unit of sound in speech, and therefore this is an important step when it comes to trying to achieve accurate pronunciation. It allows the program to map written words onto their corresponding spoken sounds.
- We can achieve and demonstrate basic phonetic conversion using the phonemizer library. However, it is **extremely important to note** that production TTS systems (e.g. ElevenLabs, Tacotron, StyleTTS2, Amazon Polly, FastSpeech2, etc) and other TTS systems (Kokoro-82m, Dia-1.6b) tend to use more complex transformer-based grapheme-to-phoneme (G2P, remember this abbreviation!) models that consider context and language-specific rules, giving their outputs more nuance and accuracy/ability.
  - Graphemes are the smallest units of written language that represent a sound (for example, letters or letter groups in English like "c", "sh", or "ough"). These small individual sounds (that still exist as English romanized letters (or whatever the script is for the language you're working with)), get converted to individual phonemes.
- Here's a code example using the basic phonemizer library:

In [ ]:
import phonemizer
# Phonetic transcription of the text
text = "Hello, world!"
phonemes = phonemizer.phonemize(text)

print("Phonetic Transcription:", phonemes)

We will go over how these sophisticated G2P models work in-depth in a separate notebook in this section of the repo.

### Prosody Modeling
- In traditional TTS systems, prosody modeling was rule-based. Prosody was added separately to make speech sound more natural by adjusting pinch, stress and intonation. These are the elements that are so core and important to producting audio output that sounds like actual human spoken voice. However, modern neural TTS engines are much more advanced, and use an end-to-end method.